# AAVAIL Revenue Prediction: EDA & Model Evaluation
This notebook covers the Exploratory Data Analysis (EDA) and performance evaluation (comparing candidate models against a baseline) for the AAVAIL revenue prediction capstone.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Exploratory Data Analysis (EDA)

In [ ]:
# Load synthetic time-series data matching the dataset distribution
dates = pd.date_range(start="2017-11-01", periods=500, freq="D")
np.random.seed(42)

df_uk = pd.DataFrame({
    "date": dates,
    "country": "United Kingdom",
    "revenue": np.sin(np.linspace(0, 20, 500)) * 200 + 800 + np.random.normal(0, 50, 500),
    "purchases": np.random.randint(10, 50, 500),
    "total_views": np.random.randint(100, 500, 500)
})

# Visualize Time-Series Revenue Trend
plt.figure(figsize=(12, 5))
plt.plot(df_uk['date'], df_uk['revenue'], label='Daily Revenue ($)', color='navy')
plt.title('AAVAIL Daily Revenue Trend over Time (EDA)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.legend()
plt.show()

## 2. Feature Engineering & Model Training

In [ ]:
# Create 7-day lag features
for i in range(1, 8):
    df_uk[f'lag_{i}'] = df_uk['revenue'].shift(i)

data = df_uk.dropna().reset_index(drop=True)
X = data[[f'lag_{i}' for i in range(1, 8)]]
y = data['revenue']

# 80/20 Train-Test Split
split = int(len(data) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# Train Baseline Model (Simple Linear Regression / Moving Average)
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

# Train Production Model (Random Forest Regressor)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

## 3. Model vs. Baseline Visual Comparison

In [ ]:
# Plot Model vs Baseline comparison
plt.figure(figsize=(14, 6))
plt.plot(y_test.values[:50], label='Actual Revenue', color='black', linewidth=2)
plt.plot(y_pred_baseline[:50], label='Baseline Model (Linear)', color='red', linestyle='--')
plt.plot(y_pred_rf[:50], label='Final Model (Random Forest)', color='green', linestyle='-')

plt.title('Baseline Model vs. Final Model Performance Comparison', fontsize=14)
plt.xlabel('Days (Test Period)')
plt.ylabel('Revenue')
plt.legend()
plt.show()

rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
print(f'Baseline RMSE: {rmse_baseline:.2f}')
print(f'Random Forest RMSE: {rmse_rf:.2f}')